# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print(metadata['name'])
print(metadata['description'])

# Print general metadata fields
print(f"Dataset ID: {metadata['@id']}")
print(f"Dataset version: {metadata['version']}")
print(f"Published on: {metadata['datePublished']}")
print(f"Keywords: {', '.join(metadata['keywords'])}")
print(f"License: {metadata['license']}")

## 2. Data Overview
Review available record sets and fields. All entities are referenced by their `@id`.

In [ ]:
# List available record sets in the dataset
record_sets = dataset.metadata.record_sets
print('Available record sets in the dataset:')
for rs in record_sets:
    print(f"- Name: {rs.name}, @id: {rs['@id']}")

# For each record set, list available fields/columns
for rs in record_sets:
    print(f"\nRecord set '{rs.name}' (ID: {rs['@id']}):")
    print("Fields (by @id):")
    for fld in rs.fields:
        col_ids = [c['@id'] for c in fld.columns]
        print(f"  - Field '{fld.name}' @id: {fld['@id']}, dataType: {fld.data_type}, columns: {col_ids}")

## 3. Data Extraction
Load data from relevant record set(s) into a DataFrame for analysis. All references are made using `@id`.

In [ ]:
# Prepare record set variables for extraction
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
# Show all available record set IDs for reference
print(f"Record set IDs for extraction: {record_set_ids}")

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Show columns from first record set loaded
first_rs_id = record_set_ids[0] if record_set_ids else None
if first_rs_id:
    print(f"Columns in record set {first_rs_id}: {dataframes[first_rs_id].columns.tolist()}")
    display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps.

This section demonstrates:
- Filtering records based on a threshold
- Normalizing a numeric field
- Grouping by a categorical field

All operations use fields referenced by their `@id`.

In [ ]:
# Choose a record set and fields for EDA
# List columns for reference
if first_rs_id:
    columns = dataframes[first_rs_id].columns.tolist()
    print(f"Available columns for EDA in {first_rs_id}: {columns}")

    # Example: Let's assume 'age' and 'sex' are column '@id's
    numeric_field_id = None
    group_field_id = None

    # Find likely numeric and group fields by column names containing 'age', 'interval', 'sex', etc.
    for col in columns:
        if 'age' in col.lower():
            numeric_field_id = col
        elif 'interval' in col.lower():
            numeric_field_id = col
        if 'sex' in col.lower() or 'msi' in col.lower() or 'location' in col.lower():
            group_field_id = col

    print(f"Selected numeric field @id: {numeric_field_id}")
    print(f"Selected group field @id: {group_field_id}")

    # Filtering
    if numeric_field_id:
        threshold = 50  # Example threshold for age or interval
        try:
            filtered_df = dataframes[first_rs_id][dataframes[first_rs_id][numeric_field_id] > threshold]
            print(f"Filtered records where {numeric_field_id} > {threshold}:")
            display(filtered_df.head())

            # Normalization
            filtered_df[f"{numeric_field_id}_normalized"] = (
                filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
            ) / filtered_df[numeric_field_id].std()
            print(f"Normalized {numeric_field_id} for filtered records:")
            display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

            # Grouping
            if group_field_id and group_field_id in filtered_df.columns:
                grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
                print(f"Grouped data by {group_field_id}:")
                display(grouped_df.head())
        except Exception as e:
            print(f"Error during EDA: {e}")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram for numeric field
if first_rs_id and numeric_field_id:
    plt.figure(figsize=(7,4))
    df = dataframes[first_rs_id]
    if numeric_field_id in df.columns and pd.api.types.is_numeric_dtype(df[numeric_field_id]):
        sns.histplot(df[numeric_field_id], kde=True)
        plt.title(f"Distribution of {numeric_field_id}")
        plt.xlabel(numeric_field_id)
        plt.ylabel('Frequency')
        plt.show()

# Bar plot for group field count
if first_rs_id and group_field_id:
    plt.figure(figsize=(7,4))
    df = dataframes[first_rs_id]
    if group_field_id in df.columns:
        grp_counts = df[group_field_id].value_counts()
        sns.barplot(x=grp_counts.index, y=grp_counts.values)
        plt.title(f"Record counts by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel('Count')
        plt.show()

## 6. Conclusion
This notebook has demonstrated how to load, explore, and process the FAIR² dataset using the `mlcroissant` library, referencing entities by their `@id` throughout.

- We reviewed the dataset metadata and structure.
- We loaded and previewed records, referencing each record set, field, and column by its unique `@id`.
- We applied basic exploratory analysis and visualization.

Further analysis can include more complex statistical modeling and integration with clinical predictors using the robust structure provided by the Croissant schema.